# 4. Model Eğitimi

Bu notebook'ta yıldırım tahmini için makine öğrenmesi modelleri eğiteceğiz.

**Modeller:**
- Lojistik Regresyon
- Random Forest
- XGBoost
- LightGBM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost yüklü değil, atlanacak.")

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False
    print("LightGBM yüklü değil, atlanacak.")

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Kütüphaneler yüklendi!')

## 4.1 Verilerin Yüklenmesi

In [ ]:
DATA_PATH = '../data/'

# Model veri setini yükle
df = pd.read_pickle(DATA_PATH + 'model_veri_seti.pkl')

# Özellik listesini yükle
with open(DATA_PATH + 'ozellik_listesi.json', 'r', encoding='utf-8') as f:
    ozellik_bilgi = json.load(f)

hedef = ozellik_bilgi['hedef']
ozellikler = ozellik_bilgi['ozellikler']

print(f"Veri seti boyutu: {df.shape}")
print(f"Hedef değişken: {hedef}")
print(f"Özellik sayısı: {len(ozellikler)}")

## 4.2 Veri Hazırlığı

In [ ]:
# Özellik ve hedef değişkenleri ayır
X = df[ozellikler]
y = df[hedef]

print(f"X boyutu: {X.shape}")
print(f"y boyutu: {y.shape}")
print(f"\nHedef değişken dağılımı:")
print(y.value_counts())
print(f"\nPozitif sınıf oranı: {y.mean()*100:.2f}%")

In [ ]:
# Train-Test bölme
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Eğitim seti: {X_train.shape[0]} örnek")
print(f"Test seti: {X_test.shape[0]} örnek")

In [ ]:
# Ölçeklendirme
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Scaler'ı kaydet
joblib.dump(scaler, DATA_PATH + 'scaler.pkl')
print("Ölçeklendirme tamamlandı!")

## 4.3 Model Eğitimi

In [ ]:
# Sonuçları saklamak için
sonuclar = {}
modeller = {}

In [ ]:
def degerlendir_model(model, X_test, y_test, model_adi):
    """Model performansını değerlendirir."""
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    sonuc = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    }
    
    print(f"\n=== {model_adi} ===")
    for metrik, deger in sonuc.items():
        if deger is not None:
            print(f"  {metrik}: {deger:.4f}")
    
    return sonuc, y_pred, y_pred_proba

### 4.3.1 Lojistik Regresyon

In [ ]:
print("Lojistik Regresyon eğitiliyor...")
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

sonuclar['Lojistik Regresyon'], lr_pred, lr_proba = degerlendir_model(
    lr_model, X_test_scaled, y_test, 'Lojistik Regresyon'
)
modeller['Lojistik Regresyon'] = lr_model

### 4.3.2 Random Forest

In [ ]:
print("Random Forest eğitiliyor...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf_model.fit(X_train, y_train)  # RF için ölçeklendirme gerekmez

sonuclar['Random Forest'], rf_pred, rf_proba = degerlendir_model(
    rf_model, X_test, y_test, 'Random Forest'
)
modeller['Random Forest'] = rf_model

### 4.3.3 XGBoost

In [ ]:
if XGB_AVAILABLE:
    print("XGBoost eğitiliyor...")
    
    # Sınıf dengesizliği için ağırlık hesapla
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    
    sonuclar['XGBoost'], xgb_pred, xgb_proba = degerlendir_model(
        xgb_model, X_test, y_test, 'XGBoost'
    )
    modeller['XGBoost'] = xgb_model
else:
    print("XGBoost atlandı (kütüphane yüklü değil)")

### 4.3.4 LightGBM

In [ ]:
if LGB_AVAILABLE:
    print("LightGBM eğitiliyor...")
    
    lgb_model = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    )
    lgb_model.fit(X_train, y_train)
    
    sonuclar['LightGBM'], lgb_pred, lgb_proba = degerlendir_model(
        lgb_model, X_test, y_test, 'LightGBM'
    )
    modeller['LightGBM'] = lgb_model
else:
    print("LightGBM atlandı (kütüphane yüklü değil)")

## 4.4 Model Karşılaştırması

In [ ]:
# Sonuçları DataFrame'e çevir
sonuc_df = pd.DataFrame(sonuclar).T
sonuc_df = sonuc_df.round(4)

print("\n=== MODEL KARŞILAŞTIRMASI ===")
display(sonuc_df)

In [ ]:
# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrik karşılaştırma
metrikler = ['accuracy', 'precision', 'recall', 'f1']
sonuc_df[metrikler].plot(kind='bar', ax=axes[0], colormap='viridis')
axes[0].set_title('Model Performans Metrikleri')
axes[0].set_ylabel('Değer')
axes[0].legend(loc='lower right')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# ROC-AUC karşılaştırma
if 'roc_auc' in sonuc_df.columns:
    sonuc_df['roc_auc'].plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('ROC-AUC Karşılaştırması')
    axes[1].set_ylabel('AUC')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../tez_docs/figures/model_karsilastirma.png', dpi=150)
plt.show()

## 4.5 Cross-Validation

In [ ]:
# En iyi model için cross-validation
en_iyi_model_adi = sonuc_df['f1'].idxmax()
en_iyi_model = modeller[en_iyi_model_adi]

print(f"En iyi model (F1 skoruna göre): {en_iyi_model_adi}")

# 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

if en_iyi_model_adi == 'Lojistik Regresyon':
    X_cv = X_train_scaled
else:
    X_cv = X_train

cv_scores = cross_val_score(en_iyi_model, X_cv, y_train, cv=cv, scoring='f1')

print(f"\n5-Fold CV Sonuçları (F1):")
print(f"  Ortalama: {cv_scores.mean():.4f}")
print(f"  Std: {cv_scores.std():.4f}")
print(f"  Fold skorları: {cv_scores}")

## 4.6 Modellerin Kaydedilmesi

In [ ]:
# Tüm modelleri kaydet
for model_adi, model in modeller.items():
    dosya_adi = model_adi.lower().replace(' ', '_') + '_model.pkl'
    joblib.dump(model, DATA_PATH + dosya_adi)
    print(f"Kaydedildi: {dosya_adi}")

# Sonuçları kaydet
sonuc_df.to_pickle(DATA_PATH + 'model_sonuclari.pkl')
sonuc_df.to_csv(DATA_PATH + 'model_sonuclari.csv')

print("\nTüm modeller ve sonuçlar kaydedildi!")

---
**Sonraki Adım:** `05_sonuclarin_analizi.ipynb` - Sonuçların Detaylı Analizi